# Quick Pilot: AlphaEarth + XAI (EU Cross-country)

目标：用最少数据快速验证“模型是否学到国家shortcut而非作物语义”。

## 最小研究计划（3-5小时可跑完）
- 数据：先取 3 个国家（建议 DE/BE/ES），每国 1 年，最多各 5k 样本
- 实验A（ID）：随机划分训练/测试
- 实验B（OOD）：留一国测试（train on 2 countries, test on 1 country）
- 模型：LightGBM（或 RandomForest 兜底）
- XAI：Permutation Importance +（可选）SHAP
- 判据：如果 OOD 相比 ID 大幅下降，且“国家字段/代理变量”重要性高，则存在shortcut风险

In [ ]:
# 安装依赖（Colab）
!pip -q install pandas numpy scikit-learn lightgbm shap pyarrow

In [ ]:
from google.colab import drive
import os, re, glob, json, zipfile, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# === 修改为你的项目路径 ===
BASE_DIR = '/content/drive/MyDrive/Research/Alpha_Crops'
DATA_DIR = os.path.join(BASE_DIR, '14094196')
OUT_DIR = os.path.join(BASE_DIR, 'results', 'quick_pilot')
os.makedirs(OUT_DIR, exist_ok=True)

print('BASE_DIR exists:', os.path.exists(BASE_DIR))
print('DATA_DIR exists:', os.path.exists(DATA_DIR))

In [ ]:
# 列出可用国家文件
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
zip_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.zip')))
print('csv files:', len(csv_files), ' zip files:', len(zip_files))
for p in csv_files[:30]:
    print('-', os.path.basename(p))

In [ ]:
# 若csv打不开/不存在，可从zip解压需要的国家（按需）
NEEDED = ['DE', 'BE', 'ES']
for z in zip_files:
    name = os.path.basename(z).upper()
    if any(code in name for code in NEEDED):
        try:
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(DATA_DIR)
            print('unzipped:', os.path.basename(z))
        except Exception as e:
            print('skip unzip:', os.path.basename(z), e)

In [ ]:
# 读取每个国家一个CSV，自动推断label和特征列
country_patterns = {
    'DE': r'^de',
    'BE': r'^be',
    'ES': r'^es'
}

def pick_file(country):
    pats = [p for p in csv_files if re.search(country_patterns[country], os.path.basename(p), re.I)]
    if len(pats)==0:
        pats = [p for p in glob.glob(os.path.join(DATA_DIR,'*.csv')) if country.lower() in os.path.basename(p).lower()]
    return pats[0] if pats else None

frames = []
for c in ['DE','BE','ES']:
    fp = pick_file(c)
    print(c, '->', fp)
    if not fp: 
        continue
    df = pd.read_csv(fp, low_memory=False)
    df['country'] = c
    frames.append(df)

if not frames:
    raise RuntimeError('未找到可读取CSV，请检查Drive路径和文件是否已本地可用（iCloud/Drive同步状态）。')

df = pd.concat(frames, ignore_index=True)
print('raw shape:', df.shape)

label_candidates = ['class', 'crop', 'crop_code', 'label', 'HCAT2', 'HCAT3', 'ec_hcat2', 'ec_hcat3']
label_col = None
for c in label_candidates:
    if c in df.columns:
        label_col = c
        break
if label_col is None:
    # 兜底：找低基数文本列
    for c in df.columns:
        if df[c].dtype == 'object' and df[c].nunique(dropna=True) > 2 and df[c].nunique(dropna=True) < 200:
            label_col = c
            break

if label_col is None:
    raise RuntimeError('自动识别label失败，请手动设置label_col。')

print('label_col =', label_col)

# 特征：优先 AlphaEarth embedding A00-A63；否则用数值列
ae_cols = [c for c in df.columns if re.fullmatch(r'A\d{2}', str(c))]
if len(ae_cols) >= 8:
    feat_cols = ae_cols
else:
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    drop_cols = set([label_col])
    feat_cols = [c for c in num_cols if c not in drop_cols][:128]

print('feature cols:', len(feat_cols))

use_cols = feat_cols + [label_col, 'country']
df = df[use_cols].dropna()

# 每国每类最多采样500，确保快速
SAMPLE_PER_CLASS_COUNTRY = 500
parts = []
for (cty, y), g in df.groupby(['country', label_col]):
    parts.append(g.sample(min(len(g), SAMPLE_PER_CLASS_COUNTRY), random_state=42))
df_small = pd.concat(parts, ignore_index=True)
print('sampled shape:', df_small.shape)

In [ ]:
# 实验A：ID随机划分
X = df_small[feat_cols]
y = df_small[label_col].astype(str)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

try:
    from lightgbm import LGBMClassifier
    clf_id = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, class_weight='balanced', random_state=42)
except Exception:
    clf_id = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)

clf_id.fit(X_train, y_train)
pred_id = clf_id.predict(X_test)
f1_id = f1_score(y_test, pred_id, average='macro')
print('ID macro-F1 =', round(float(f1_id), 4))

In [ ]:
# 实验B：OOD 留一国测试（DE+BE -> ES，可自行循环）
train_cty = ['DE','BE']
test_cty = 'ES'

tr = df_small[df_small['country'].isin(train_cty)]
te = df_small[df_small['country']==test_cty]

Xtr, ytr = tr[feat_cols], tr[label_col].astype(str)
Xte, yte = te[feat_cols], te[label_col].astype(str)

# 仅保留训练中出现的类
keep = set(ytr.unique())
mask = yte.isin(keep)
Xte, yte = Xte[mask], yte[mask]

try:
    from lightgbm import LGBMClassifier
    clf_ood = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, class_weight='balanced', random_state=42)
except Exception:
    clf_ood = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)

clf_ood.fit(Xtr, ytr)
pred_ood = clf_ood.predict(Xte)
f1_ood = f1_score(yte, pred_ood, average='macro') if len(yte)>0 else np.nan
print('OOD macro-F1 =', round(float(f1_ood), 4) if pd.notna(f1_ood) else 'nan')

summary = {
    'id_macro_f1': float(f1_id),
    'ood_macro_f1': float(f1_ood) if pd.notna(f1_ood) else None,
    'drop': float(f1_id - f1_ood) if pd.notna(f1_ood) else None,
    'n_train': int(len(tr)),
    'n_test_ood': int(len(yte)),
    'label_col': label_col,
    'n_features': int(len(feat_cols))
}
print(summary)
with open(os.path.join(OUT_DIR, 'quick_pilot_summary.json'),'w') as f:
    json.dump(summary, f, indent=2)

In [ ]:
# XAI: permutation importance（快速稳健）
from sklearn.metrics import make_scorer, f1_score
scorer = make_scorer(f1_score, average='macro')

sample_idx = np.random.RandomState(42).choice(len(Xte), size=min(2000, len(Xte)), replace=False) if len(Xte)>0 else []
if len(sample_idx)>0:
    r = permutation_importance(clf_ood, Xte.iloc[sample_idx], yte.iloc[sample_idx], n_repeats=5, random_state=42, scoring=scorer)
    imp = pd.DataFrame({'feature': feat_cols, 'importance': r.importances_mean}).sort_values('importance', ascending=False)
    print(imp.head(20))
    imp.to_csv(os.path.join(OUT_DIR, 'perm_importance_top.csv'), index=False)
else:
    print('OOD测试样本为空，跳过XAI。')

## 汇报模板（复制到群里）
- Pilot完成：DE+BE训练，ES测试
- ID macro-F1: `xx`
- OOD macro-F1: `yy`
- Gap: `xx-yy`
- 结论：若Gap显著且某些特征在跨国时主导，说明存在shortcut风险；下一步做country-invariant训练与层级标签对齐。